# UNet Lab: modify and test the UNet architecture

A workbench for **changing UNet and measuring what each change does**. It follows the classic UNet explanation
(the original paper, as in the GeeksforGeeks article *"U-Net Architecture Explained"*) and turns every part of the diagram into a setting you can change.

```
 input ─► [block]──────────── skip (copy & crop) ───────────────►[merge]─►[block]─► 1×1 conv ─► mask
             │ down                                                 ▲ up
             ▼                                                      │
           [block]─────────── skip ──────────────────►[merge]─►[block]
             │ down                                     ▲ up
             ▼                                          │
           [block]──── skip ────────►[merge]─►[block]
             │ down                   ▲ up
             ▼                        │
           [block]── skip ─►[merge]─►[block]
             │ down          ▲ up
             ▼               │
          [bottleneck] ──────┘ (+ dropout)
   CONTRACTING PATH          EXPANSIVE PATH
```

| Part | What you do |
|---|---|
| **1** | Build the **original UNet** from the paper and trace every shape (572 → 388) |
| **2** | `UNetLab`: the same network with **every part as a setting**, plus a map from the diagram to the settings |
| **3** | **Test bench**: params, GFLOPs, GPU memory and speed for many variants at once |
| **4** | **Quick check**: can each variant learn one COD10K image? (about 1 minute) |
| **5** | **Real test**: train your chosen variants on COD10K and compare them in one table and plot |
| **6** | **Add your own modification**: a template for plugging in a new block or skip gate |

> **Note:** this lab trains UNet **from scratch** (no pretrained encoder), like the paper.
> Expect lower scores than your pretrained ResNet50-UNet (test MAE 0.039). The point here is the **relative** effect of each change.

## Setup

In [ ]:
import os
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from blocks import count_params, show, to_tensor
from config import IMAGE_SIZE, OUTPUT_DIR, get_device
from data import CODDataset, make_splits
from train_utils import append_result, bce_iou_loss, evaluate, train_one_epoch

device = get_device()
print(f"device: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

CHECKPOINT_DIR = os.path.join(os.path.dirname(OUTPUT_DIR), "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

## Part 1: the original UNet, exactly as in the paper

The article's key points, and where they appear in the code:

| article / paper | code |
|---|---|
| **Contracting path**: repeated two 3×3 convs + ReLU, then 2×2 max-pool; channels double (64 → 1024) | `down1 … down4`, `self.pool` |
| **Unpadded** ("valid") convs: each one loses 2 pixels | `nn.Conv2d(cin, cout, 3)` has `padding=0` |
| **Bottleneck** at the bottom (1024 channels) | `self.bottleneck` |
| **Expansive path**: 2×2 **up-convolution** halves the channels | `nn.ConvTranspose2d(…, 2, stride=2)` |
| **Copy and crop**: the encoder feature is cropped to match, then **concatenated** | `self.crop(...)`, `torch.cat` |
| **1×1 conv** maps 64 channels to the classes | `self.out` |
| Input **572×572** → output **388×388** | checked below |

In [ ]:
class PaperUNet(nn.Module):
    """
    The ORIGINAL UNet (Ronneberger et al., 2015), exactly as in the paper / article:
      - 3x3 convs with NO padding ("valid"), so every conv shrinks H and W by 2
      - ReLU, no BatchNorm
      - 2x2 max-pool for down, 2x2 transposed conv ("up-conv") for up
      - "copy and crop": the encoder feature is centre-cropped to the decoder's size before concat
      - 1x1 conv at the end -> 2 classes
    Input 572x572 -> output 388x388.
    """

    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()

        def double_conv(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3), nn.ReLU(inplace=True),    # padding=0 by default -> "valid"
                nn.Conv2d(cout, cout, 3), nn.ReLU(inplace=True),
            )

        # contracting path (encoder)
        self.down1 = double_conv(in_channels, 64)
        self.down2 = double_conv(64, 128)
        self.down3 = double_conv(128, 256)
        self.down4 = double_conv(256, 512)
        self.pool = nn.MaxPool2d(2)
        # bottleneck
        self.bottleneck = double_conv(512, 1024)
        # expansive path (decoder): up-conv halves the channels, then concat doubles them again
        self.upconv4, self.up4 = nn.ConvTranspose2d(1024, 512, 2, stride=2), double_conv(1024, 512)
        self.upconv3, self.up3 = nn.ConvTranspose2d(512, 256, 2, stride=2), double_conv(512, 256)
        self.upconv2, self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2), double_conv(256, 128)
        self.upconv1, self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2), double_conv(128, 64)
        # output
        self.out = nn.Conv2d(64, num_classes, 1)

    @staticmethod
    def crop(skip, x):
        """Centre-crop the encoder feature `skip` to the spatial size of `x`."""
        H, W = skip.shape[-2:]
        h, w = x.shape[-2:]
        top, left = (H - h) // 2, (W - w) // 2
        return skip[..., top:top + h, left:left + w]

    def forward(self, x):
        s1 = self.down1(x)
        s2 = self.down2(self.pool(s1))
        s3 = self.down3(self.pool(s2))
        s4 = self.down4(self.pool(s3))
        b = self.bottleneck(self.pool(s4))

        x = self.upconv4(b)
        x = self.up4(torch.cat([self.crop(s4, x), x], dim=1))
        x = self.upconv3(x)
        x = self.up3(torch.cat([self.crop(s3, x), x], dim=1))
        x = self.upconv2(x)
        x = self.up2(torch.cat([self.crop(s2, x), x], dim=1))
        x = self.upconv1(x)
        x = self.up1(torch.cat([self.crop(s1, x), x], dim=1))
        return self.out(x)

In [ ]:
paper = PaperUNet(in_channels=1, num_classes=2)
print(f"parameters: {count_params(paper):,}   (the paper: about 31M)\n")

# Trace every shape with forward hooks: a hook is a function PyTorch calls after a layer runs.
trace = []
for name in ["down1", "down2", "down3", "down4", "bottleneck", "upconv4", "up4", "upconv3", "up3",
             "upconv2", "up2", "upconv1", "up1", "out"]:
    getattr(paper, name).register_forward_hook(lambda m, i, o, name=name: trace.append((name, tuple(o.shape[1:]))))

with torch.no_grad():
    y = paper(torch.randn(1, 1, 572, 572))
for name, shape in trace:
    print(f"{name:<11} -> {shape}")
print(f"\ninput (1, 572, 572)  ->  output {tuple(y.shape[1:])}")

**Why crop?** Every unpadded 3×3 conv loses a 1-pixel border, so the encoder feature `s4` (64×64) is bigger than the upsampled decoder feature (56×56).
The paper cuts out the centre of `s4` to match. Because of that, the output (388) is smaller than the input (572).
The paper handled it with an "overlap-tile" strategy (predict tiles of a bigger, mirror-padded image).

**Modern UNets use `padding=1`** ("same") instead, so input and output sizes match and no crop is needed. That's the first modification you'll test.

> ✏️ **TRY IT**: feed `torch.randn(1, 1, 256, 256)`. The paper's UNet only works for some input sizes. Why? (Hint: every level must leave an even size for the max-pool.)

## Part 2: `UNetLab`, UNet with every part as a setting

The building parts come first. Each has a **registry** (a dict) so you can add your own later:

| registry / function | diagram part | options |
|---|---|---|
| `BLOCKS` | the **[block]** boxes and the **bottleneck** | `double` (paper), `residual`, `dws` (depthwise-separable), `dilated` |
| `make_down` | the **down** arrows | `maxpool` (paper), `avgpool`, `conv` (learnable, stride 2) |
| `make_up` | the **up** arrows | `transpose` (paper's up-conv), `bilinear`, `nearest` |
| `GATES` | what happens to the **skip** before merging | `none` (paper), `attention` (Attention U-Net), `se` (channel attention) |
| `merge` | the **[merge]** boxes | `concat` (paper), `add`, `none` (no skips at all) |
| `ConvNormAct` | inside every conv | `padding` same/valid, `norm` bn/gn/none, `act` relu/leakyrelu/gelu/silu |

In [ ]:
# ---------------------------------------------------------------------------
# Small helpers
# ---------------------------------------------------------------------------
def center_crop(t, size):
    """Centre-crop t [B, C, H, W] to spatial `size` (the paper's "copy and crop")."""
    H, W = t.shape[-2:]
    h, w = size
    top, left = (H - h) // 2, (W - w) // 2
    return t[..., top:top + h, left:left + w]


def make_norm(kind, ch):
    if kind == "bn":
        return nn.BatchNorm2d(ch)
    if kind == "gn":
        return nn.GroupNorm(8 if ch % 8 == 0 else 1, ch)
    return nn.Identity()          # "none" (the original paper)


ACTIVATIONS = {
    "relu": lambda: nn.ReLU(inplace=True),
    "leakyrelu": lambda: nn.LeakyReLU(0.1, inplace=True),
    "gelu": lambda: nn.GELU(),
    "silu": lambda: nn.SiLU(inplace=True),
    "none": lambda: nn.Identity(),
}


class ConvNormAct(nn.Sequential):
    """conv -> norm -> activation. `padding` is "same" (keep H, W) or "valid" (paper: shrink by k-1)."""

    def __init__(self, cin, cout, k=3, stride=1, dilation=1, groups=1, padding="same", norm="bn", act="relu"):
        pad = dilation * (k - 1) // 2 if padding == "same" else 0
        super().__init__(
            nn.Conv2d(cin, cout, k, stride=stride, padding=pad, dilation=dilation, groups=groups,
                      bias=(norm == "none")),
            make_norm(norm, cout),
            ACTIVATIONS[act](),
        )


# ---------------------------------------------------------------------------
# BLOCKS: what each encoder/decoder stage is made of.  Signature: Block(cin, cout, cfg)
# ---------------------------------------------------------------------------
class DoubleConvBlock(nn.Module):
    """The UNet block: two 3x3 conv-norm-act."""

    def __init__(self, cin, cout, cfg):
        super().__init__()
        p, n, a = cfg["padding"], cfg["norm"], cfg["act"]
        self.body = nn.Sequential(ConvNormAct(cin, cout, padding=p, norm=n, act=a),
                                  ConvNormAct(cout, cout, padding=p, norm=n, act=a))

    def forward(self, x):
        return self.body(x)


class ResidualBlock(nn.Module):
    """Two convs + a shortcut (ResNet idea). The shortcut is cropped if 'valid' padding shrank the body."""

    def __init__(self, cin, cout, cfg):
        super().__init__()
        p, n, a = cfg["padding"], cfg["norm"], cfg["act"]
        self.body = nn.Sequential(ConvNormAct(cin, cout, padding=p, norm=n, act=a),
                                  ConvNormAct(cout, cout, padding=p, norm=n, act="none"))
        self.shortcut = nn.Identity() if cin == cout else ConvNormAct(cin, cout, k=1, norm=n, act="none")
        self.act = ACTIVATIONS[a]()

    def forward(self, x):
        out = self.body(x)
        return self.act(out + center_crop(self.shortcut(x), out.shape[-2:]))


class DepthwiseSeparableBlock(nn.Module):
    """Two depthwise-separable convs (MobileNet idea): far fewer parameters."""

    def __init__(self, cin, cout, cfg):
        super().__init__()
        p, n, a = cfg["padding"], cfg["norm"], cfg["act"]
        self.body = nn.Sequential(
            ConvNormAct(cin, cin, groups=cin, padding=p, norm=n, act=a),    # depthwise 3x3
            ConvNormAct(cin, cout, k=1, norm=n, act=a),                      # pointwise 1x1
            ConvNormAct(cout, cout, groups=cout, padding=p, norm=n, act=a),
            ConvNormAct(cout, cout, k=1, norm=n, act=a),
        )

    def forward(self, x):
        return self.body(x)


class DilatedBlock(nn.Module):
    """Parallel 3x3 convs with dilation 1, 2, 4, 8, summed: sees small AND large context (like SINet's RF module)."""

    def __init__(self, cin, cout, cfg):
        super().__init__()
        n, a = cfg["norm"], cfg["act"]
        self.branches = nn.ModuleList([ConvNormAct(cin, cout, dilation=d, padding="same", norm=n, act=a)
                                       for d in (1, 2, 4, 8)])
        self.fuse = ConvNormAct(cout, cout, k=1, norm=n, act=a)

    def forward(self, x):
        return self.fuse(sum(b(x) for b in self.branches))


BLOCKS = {
    "double": DoubleConvBlock,
    "residual": ResidualBlock,
    "dws": DepthwiseSeparableBlock,
    "dilated": DilatedBlock,
}


# ---------------------------------------------------------------------------
# DOWN: how the contracting path halves H and W
# ---------------------------------------------------------------------------
def make_down(kind, ch, cfg):
    if kind == "maxpool":
        return nn.MaxPool2d(2)
    if kind == "avgpool":
        return nn.AvgPool2d(2)
    if kind == "conv":
        return ConvNormAct(ch, ch, k=3, stride=2, padding="same", norm=cfg["norm"], act=cfg["act"])
    raise ValueError(f"unknown down: {kind}")


# ---------------------------------------------------------------------------
# UP: how the expansive path doubles H and W (and sets the channels to cout)
# ---------------------------------------------------------------------------
def make_up(kind, cin, cout):
    if kind == "transpose":                    # the paper's 2x2 "up-conv"
        return nn.ConvTranspose2d(cin, cout, 2, stride=2)
    if kind == "bilinear":
        return nn.Sequential(nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False), nn.Conv2d(cin, cout, 1))
    if kind == "nearest":
        return nn.Sequential(nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(cin, cout, 1))
    raise ValueError(f"unknown up: {kind}")


# ---------------------------------------------------------------------------
# SKIP GATES: modify the encoder feature before it is merged.  Signature: Gate(skip_ch, gate_ch)
# forward(skip, g) where g is the upsampled decoder feature at the same scale.
# ---------------------------------------------------------------------------
class NoGate(nn.Module):
    def __init__(self, skip_ch, gate_ch):
        super().__init__()

    def forward(self, skip, g):
        return skip


class AttentionGate(nn.Module):
    """
    Attention U-Net (Oktay et al., 2018). The decoder feature g decides WHERE the skip is useful:
    a = sigmoid(psi(relu(Wx*skip + Wg*g)))  ->  skip * a   (a is one weight per pixel, 0..1)
    """

    def __init__(self, skip_ch, gate_ch):
        super().__init__()
        inter = max(skip_ch // 2, 8)
        self.wx = nn.Conv2d(skip_ch, inter, 1)
        self.wg = nn.Conv2d(gate_ch, inter, 1)
        self.psi = nn.Sequential(nn.ReLU(inplace=True), nn.Conv2d(inter, 1, 1), nn.Sigmoid())
        self.last_attention = None

    def forward(self, skip, g):
        a = self.psi(self.wx(skip) + self.wg(g))
        self.last_attention = a.detach()         # kept so we can look at it
        return skip * a


class SEGate(nn.Module):
    """Squeeze-and-Excitation on the skip: decides WHICH CHANNELS of the skip are useful (one weight per channel)."""

    def __init__(self, skip_ch, gate_ch, reduction=8):
        super().__init__()
        hidden = max(skip_ch // reduction, 4)
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(skip_ch, hidden, 1), nn.ReLU(inplace=True),
                                nn.Conv2d(hidden, skip_ch, 1), nn.Sigmoid())

    def forward(self, skip, g):
        return skip * self.fc(skip)


GATES = {
    "none": NoGate,
    "attention": AttentionGate,
    "se": SEGate,
}

Now the network itself. `DEFAULT_CFG` lists **every setting** with a comment. `UNetLab(**changes)` builds a UNet with your changes applied,
for example `UNetLab(skip_gate="attention", base_channels=32)`.

In [ ]:
DEFAULT_CFG = dict(
    in_channels=3, num_classes=1,
    depth=5,               # number of levels (the paper: 5 = 4 down-steps + bottleneck)
    base_channels=64,      # channels at level 1 (the paper: 64)
    channel_mult=2,        # channels multiply by this at every level (64, 128, 256, 512, 1024)
    padding="same",        # "same" keeps H,W | "valid" = the paper (convs shrink, skips are cropped)
    block="double",        # a key of BLOCKS
    bottleneck="double",   # a key of BLOCKS, used only for the deepest level
    norm="bn",             # "bn" | "gn" | "none" (the paper)
    act="relu",            # a key of ACTIVATIONS
    down="maxpool",        # "maxpool" (paper) | "avgpool" | "conv"
    up="bilinear",         # "transpose" (paper) | "bilinear" | "nearest"
    merge="concat",        # "concat" (paper) | "add" | "none" (no skip connections)
    skip_gate="none",      # a key of GATES
    dropout=0.0,           # Dropout2d after the bottleneck (the paper used 0.5)
    resize_output=True,    # resize the output to the input size (needed for "valid" padding in training)
)


class DecoderStep(nn.Module):
    """One step of the expansive path: up -> (gate the skip) -> merge -> block."""

    def __init__(self, cin, skip_ch, cout, cfg):
        super().__init__()
        self.merge = cfg["merge"]
        self.up = make_up(cfg["up"], cin, cout)
        self.gate = GATES[cfg["skip_gate"]](skip_ch, cout)
        if self.merge == "concat":
            block_in = cout + skip_ch
        elif self.merge == "add":
            self.proj = nn.Conv2d(skip_ch, cout, 1)
            block_in = cout
        else:
            block_in = cout
        self.block = BLOCKS[cfg["block"]](block_in, cout, cfg)

    def forward(self, x, skip):
        x = self.up(x)
        if self.merge != "none":
            if skip.shape[-2] >= x.shape[-2] and skip.shape[-1] >= x.shape[-1]:
                skip = center_crop(skip, x.shape[-2:])                  # "copy and crop"
            else:
                x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
            skip = self.gate(skip, x)
            x = torch.cat([skip, x], dim=1) if self.merge == "concat" else x + self.proj(skip)
        return self.block(x)


class UNetLab(nn.Module):
    """A UNet where every design choice is an argument. UNetLab(**changes) overrides DEFAULT_CFG."""

    def __init__(self, **changes):
        super().__init__()
        unknown = set(changes) - set(DEFAULT_CFG)
        if unknown:
            raise ValueError(f"unknown settings: {unknown}")
        cfg = {**DEFAULT_CFG, **changes}
        self.cfg = cfg
        ch = [int(cfg["base_channels"] * cfg["channel_mult"] ** i) for i in range(cfg["depth"])]
        self.channels = ch

        # contracting path: levels 1 .. depth-1
        self.downs, self.enc = nn.ModuleList(), nn.ModuleList()
        prev = cfg["in_channels"]
        for i in range(cfg["depth"] - 1):
            self.downs.append(nn.Identity() if i == 0 else make_down(cfg["down"], prev, cfg))
            self.enc.append(BLOCKS[cfg["block"]](prev, ch[i], cfg))
            prev = ch[i]

        # bottleneck: the deepest level
        self.bottom_down = make_down(cfg["down"], prev, cfg)
        self.bottleneck = BLOCKS[cfg["bottleneck"]](prev, ch[-1], cfg)
        self.dropout = nn.Dropout2d(cfg["dropout"]) if cfg["dropout"] > 0 else nn.Identity()

        # expansive path: back up, one DecoderStep per encoder level
        self.dec = nn.ModuleList([DecoderStep(ch[i + 1], ch[i], ch[i], cfg) for i in reversed(range(cfg["depth"] - 1))])

        # 1x1 conv to the classes
        self.head = nn.Conv2d(ch[0], cfg["num_classes"], 1)

    def forward(self, x):
        size = x.shape[-2:]
        skips = []
        for down, block in zip(self.downs, self.enc):
            x = block(down(x))
            skips.append(x)
        x = self.dropout(self.bottleneck(self.bottom_down(x)))
        for step, skip in zip(self.dec, reversed(skips)):
            x = step(x, skip)
        x = self.head(x)
        if self.cfg["resize_output"] and x.shape[-2:] != size:
            x = F.interpolate(x, size=size, mode="bilinear", align_corners=False)
        return x

In [ ]:
# Check: UNetLab configured like the paper must match PaperUNet exactly (same shape, same parameter count)
paper_cfg = dict(in_channels=1, num_classes=2, padding="valid", norm="none", up="transpose", resize_output=False)
lab_as_paper = UNetLab(**paper_cfg)
with torch.no_grad():
    y = lab_as_paper(torch.randn(1, 1, 572, 572))
print(f"UNetLab(paper settings): output {tuple(y.shape[1:])}   params {count_params(lab_as_paper):,}")
print(f"PaperUNet              : output (2, 388, 388)   params {count_params(paper):,}")

## Part 3: test bench

Describe each variant as **only the settings that differ** from the defaults.
`bench()` builds each one and measures:

- **params**: how many weights (model size on disk)
- **GFLOPs**: how much computation for one 352×352 image (speed on any hardware)
- **train memory / time**: one real training step at batch 8 with AMP on your GPU (does it fit in 6 GB?)

✏️ Add, remove or change variants in `VARIANTS` and re-run. This is your playground.

In [ ]:
BASE = dict(base_channels=32)   # the starting point for every variant below (32 keeps training fast on 6 GB)

VARIANTS = {
    "baseline (modern UNet)":   dict(),
    "paper-style":              dict(padding="valid", norm="none", up="transpose"),
    "no BatchNorm":             dict(norm="none"),
    "GroupNorm + GELU":         dict(norm="gn", act="gelu"),
    "wider (base 64)":          dict(base_channels=64),
    "slimmer (x1.5 per level)": dict(channel_mult=1.5),
    "shallower (depth 4)":      dict(depth=4),
    "deeper (depth 6)":         dict(depth=6),
    "residual blocks":          dict(block="residual"),
    "depthwise-separable":      dict(block="dws"),
    "dilated bottleneck":       dict(bottleneck="dilated"),
    "strided-conv down":        dict(down="conv"),
    "transpose up":             dict(up="transpose"),
    "add instead of concat":    dict(merge="add"),
    "no skip connections":      dict(merge="none"),
    "attention gates":          dict(skip_gate="attention"),
    "SE on skips":              dict(skip_gate="se"),
}

In [ ]:
from torch.utils.flop_counter import FlopCounterMode


def build(changes):
    return UNetLab(**{**BASE, **changes})


def gflops(model, size=IMAGE_SIZE):
    model.eval()
    x = torch.randn(1, 3, size, size, device=next(model.parameters()).device)
    with FlopCounterMode(display=False) as counter, torch.no_grad():
        model(x)
    return counter.get_total_flops() / 1e9


def train_step_cost(model, batch_size=8):
    """Peak GPU memory and time of one training step (forward + backward) with AMP."""
    if device.type != "cuda":
        return float("nan"), float("nan")
    model.train()
    x = torch.randn(batch_size, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)
    for i in range(2):                               # the first run is a warm-up
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        try:
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(x).float().mean()
            loss.backward()
        except torch.OutOfMemoryError:
            model.zero_grad(set_to_none=True)
            return float("inf"), float("nan")
        torch.cuda.synchronize()
        model.zero_grad(set_to_none=True)
    return torch.cuda.max_memory_allocated() / 1024**3, (time.perf_counter() - t0) * 1000


def bench(variants):
    print(f"{'variant':<27}{'params':>11}{'GFLOPs':>9}{'train mem':>11}{'step':>9}   output")
    rows = {}
    for name, changes in variants.items():
        model = build(changes).to(device)
        with torch.no_grad():
            out = model.eval()(torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=device))
        mem, ms = train_step_cost(model)
        rows[name] = dict(params=count_params(model), gflops=gflops(model), mem=mem, ms=ms)
        mem_s = "OOM" if mem == float("inf") else ("-" if np.isnan(mem) else f"{mem:.2f} GB")
        ms_s = "-" if np.isnan(ms) else f"{ms:.0f} ms"
        print(f"{name:<27}{rows[name]['params']:>11,}{rows[name]['gflops']:>9.1f}{mem_s:>11}{ms_s:>9}   {tuple(out.shape)}")
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()
    return rows


bench_rows = bench(VARIANTS)

**How to read it**
- **Params and GFLOPs are not the same thing.** Deep levels hold most of the params (many channels), while level 1 holds most of the FLOPs and memory (big H×W).
  Look at "deeper (depth 6)" vs "wider (base 64)": almost the **same params** (≈29M each), but "wider" needs about **3× the GFLOPs**,
  because it doubles the channels at the full-resolution level too.
- **"no skip connections"** saves only about 20% of the compute (the decoder convs receive fewer channels).
  Skips are cheap, yet in Lesson 03 Part B you saw how much detail they carry.
- **Attention gates / SE** add almost nothing in params. If they help, they are "cheap wins", which is why COD papers use attention everywhere.

## Part 4: quick check, can each variant learn one image?

Before spending 20+ minutes training, check that a variant **can learn at all**: train on one COD10K image for a few steps.
If a variant can't even memorise one image, don't train it on the full dataset.

With **attention gates** we can also *see* where the gate lets the skip through (bright = kept).

In [ ]:
SANITY_VARIANTS = ["baseline (modern UNet)", "paper-style", "no skip connections", "attention gates"]  # ✏️
SANITY_STEPS = 80
SAMPLE_INDEX = 0   # ✏️ try other test images

_, _, test_pairs_all = make_splits(quick=False)
if test_pairs_all:
    image = cv2.resize(cv2.imread(test_pairs_all[SAMPLE_INDEX][0]), (IMAGE_SIZE, IMAGE_SIZE))
    mask = cv2.resize(cv2.imread(test_pairs_all[SAMPLE_INDEX][1], cv2.IMREAD_GRAYSCALE), (IMAGE_SIZE, IMAGE_SIZE),
                      interpolation=cv2.INTER_NEAREST)
else:
    print("COD10K not found -> synthetic image")
    image = cv2.add(np.full((IMAGE_SIZE, IMAGE_SIZE, 3), (60, 120, 80), np.uint8),
                    np.random.default_rng(0).integers(0, 40, (IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8))
    mask = np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.uint8)
    cv2.circle(mask, (170, 180), 80, 255, -1)
    image[mask > 0] = cv2.add(image[mask > 0], np.array([15, 20, 10], np.uint8))

x = to_tensor(image).to(device)
y = torch.from_numpy((mask > 127).astype(np.float32))[None, None].to(device)


def iou_of(prob, gt):
    p, g = prob > 0.5, gt > 127
    union = np.logical_or(p, g).sum()
    return np.logical_and(p, g).sum() / union if union else 1.0


use_amp = device.type == "cuda"
sanity = {}
for name in SANITY_VARIANTS:
    torch.manual_seed(0)
    model = build(VARIANTS[name]).to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    t0 = time.perf_counter()
    for step in range(SANITY_STEPS):
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            logits = model(x)
        loss = bce_iou_loss(logits, y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
    prob = torch.sigmoid(logits.detach().float())[0, 0].cpu().numpy()
    gates = [s.gate.last_attention[0, 0].float().cpu().numpy() for s in model.dec
             if getattr(s.gate, "last_attention", None) is not None]
    sanity[name] = dict(prob=prob, gates=gates, iou=iou_of(prob, mask), loss=loss.item())
    print(f"{name:<27} loss {loss.item():.3f}   IoU {sanity[name]['iou']:.3f}   ({time.perf_counter() - t0:.1f}s)")
    del model

In [ ]:
show([image, mask] + [s["prob"] for s in sanity.values()],
     ["image", "GT"] + [f"{n}\nIoU {s['iou']:.2f}" for n, s in sanity.items()], cmap="gray", cols=6, size=3)

for name, s in sanity.items():
    if s["gates"]:
        print(f"{name}: attention maps, deepest step -> highest resolution (bright = skip kept)")
        show(s["gates"], [f"decoder step {k + 1}  {g.shape[0]}x{g.shape[1]}" for k, g in enumerate(s["gates"])],
             cmap="viridis", cols=4, size=3)

> ✏️ **TRY IT**
> - Add `"depthwise-separable"` or `"residual blocks"` to `SANITY_VARIANTS`. Which learns fastest?
> - Does the attention map light up on the animal, or on edges? Try other `SAMPLE_INDEX` values.

## Part 5: the real test, train variants on COD10K

Pick the variants to compare in `TRAIN_VARIANTS`. Each one is trained with **identical** settings and the same data split as Lessons 04–05,
then scored on the test set and added to `outputs/results.csv`.

**Time budget (RTX 4050, base 32, batch 8):** roughly 30–60 s per epoch per variant, depending on the variant.
2 variants × 15 epochs ≈ 20–30 min. Start with `QUICK = True`.

**A fair comparison changes ONE thing.** "attention gates" vs "baseline" tells you what the gates do.
"attention gates + residual + base 64" vs "baseline" tells you nothing about which change helped.

In [ ]:
TRAIN_VARIANTS = ["baseline (modern UNet)", "attention gates"]   # ✏️ names from VARIANTS
QUICK = True                                                       # ✏️ False for the real run
EPOCHS = 2 if QUICK else 15
BATCH_SIZE = 8
LR = 1e-3            # training from scratch with BatchNorm tolerates a higher LR than fine-tuning ResNet50
NUM_WORKERS = 4
SEED = 42

train_pairs, val_pairs, test_pairs = make_splits(seed=SEED, quick=QUICK)
loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
                   persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(CODDataset(train_pairs, IMAGE_SIZE, train=True), shuffle=True, drop_last=True, **loader_args)
val_loader = DataLoader(CODDataset(val_pairs, IMAGE_SIZE), shuffle=False, **loader_args)
test_loader = DataLoader(CODDataset(test_pairs, IMAGE_SIZE), shuffle=False, **loader_args)
print(f"train {len(train_pairs)}   val {len(val_pairs)}   test {len(test_pairs)}" + ("   (QUICK subset)" if QUICK else ""))

In [ ]:
def run_name(variant):
    return "lab_" + "".join(c if c.isalnum() else "_" for c in variant.lower()).strip("_")


histories, finals = {}, {}
for variant in TRAIN_VARIANTS:
    print(f"\n=== {variant} ===")
    torch.manual_seed(SEED)
    model = build(VARIANTS[variant]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    path = os.path.join(CHECKPOINT_DIR, f"{run_name(variant)}.pth")

    hist, best_mae, best_epoch, start = [], float("inf"), -1, time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        t0 = time.perf_counter()
        loss = train_one_epoch(model, train_loader, optimizer, scaler, device)
        val = evaluate(model, val_loader, device)
        scheduler.step()
        hist.append(val["mae"])
        marker = ""
        if val["mae"] < best_mae:
            best_mae, best_epoch = val["mae"], epoch
            torch.save({"model": model.state_dict(), "cfg": model.cfg, "epoch": epoch}, path)
            marker = "  ★"
        print(f"epoch {epoch:>2}/{EPOCHS}  loss {loss:.4f}  val MAE {val['mae']:.4f}  IoU {val['iou']:.3f}"
              f"  {time.perf_counter() - t0:4.0f}s{marker}")
    minutes = (time.perf_counter() - start) / 60

    model.load_state_dict(torch.load(path, map_location=device)["model"])
    test = evaluate(model, test_loader, device)
    histories[variant] = hist
    finals[variant] = dict(test=test, best_mae=best_mae, best_epoch=best_epoch, minutes=minutes, params=count_params(model))
    append_result(os.path.join(OUTPUT_DIR, "results.csv"), {
        "run": run_name(variant), "quick": QUICK, "epochs": EPOCHS, "best_epoch": best_epoch, "batch": BATCH_SIZE,
        "lr": LR, "pretrained": False, "decoder": f"UNetLab {({**BASE, **VARIANTS[variant]})}",
        "params_M": round(count_params(model) / 1e6, 2), "val_mae": round(best_mae, 4),
        "test_mae": round(test["mae"], 4), "test_iou": round(test["iou"], 4), "test_dice": round(test["dice"], 4),
        "train_min": round(minutes, 1),
    })
    del model, optimizer
    if device.type == "cuda":
        torch.cuda.empty_cache()

In [ ]:
print(f"{'variant':<27}{'params':>11}{'test MAE':>10}{'IoU':>8}{'Dice':>8}{'best ep':>9}{'min':>7}")
for v, r in finals.items():
    t = r["test"]
    print(f"{v:<27}{r['params']:>11,}{t['mae']:>10.4f}{t['iou']:>8.3f}{t['dice']:>8.3f}{r['best_epoch']:>9}{r['minutes']:>7.1f}")

plt.figure(figsize=(7, 4))
for v, h in histories.items():
    plt.plot(range(1, len(h) + 1), h, marker="o", label=v)
plt.xlabel("epoch")
plt.ylabel("val MAE (lower is better)")
plt.title("UNet Lab: validation MAE per variant")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

**Is the difference real?** With one run per variant, a gap of a few thousandths in MAE can be pure luck (random init, data order).
Before you believe a small win, re-run both variants with another `SEED` (e.g. 1 and 2) and check that the ranking stays the same.

## Part 6: add your own modification

This is how you'll test your own ideas, for your thesis too. Three steps:

1. **Write** a new block (signature `Block(cin, cout, cfg)`) or skip gate (signature `Gate(skip_ch, gate_ch)` with `forward(skip, g)`).
2. **Register** it: `BLOCKS["my_name"] = MyBlock` or `GATES["my_name"] = MyGate`.
3. **Test** it: add a variant to `VARIANTS`, run the bench (Part 3), the quick check (Part 4), then training (Part 5).

Below is a worked example: a block with a **large 7×7 depthwise kernel** (the ConvNeXt idea), which sees more context for the same cost as a 3×3.

In [ ]:
class LargeKernelBlock(nn.Module):
    """Example custom block: 7x7 depthwise conv (big receptive field, cheap) + 1x1 pointwise, twice, with a shortcut."""

    def __init__(self, cin, cout, cfg):
        super().__init__()
        n, a = cfg["norm"], cfg["act"]
        self.inp = ConvNormAct(cin, cout, k=1, norm=n, act=a)
        self.body = nn.Sequential(
            ConvNormAct(cout, cout, k=7, groups=cout, padding="same", norm=n, act="none"),   # depthwise 7x7
            ConvNormAct(cout, cout * 2, k=1, norm=n, act=a),                                  # expand
            ConvNormAct(cout * 2, cout, k=1, norm=n, act="none"),                             # project back
        )
        self.act = ACTIVATIONS[a]()

    def forward(self, x):
        x = self.inp(x)
        return self.act(x + self.body(x))


BLOCKS["large_kernel"] = LargeKernelBlock                      # 2) register
VARIANTS["large-kernel blocks (mine)"] = dict(block="large_kernel")  # 3) add a variant

bench({k: VARIANTS[k] for k in ["baseline (modern UNet)", "large-kernel blocks (mine)"]})

> ✏️ **Ideas to try next** (each one is one registry entry + one variant):
> - A **CBAM** gate: channel attention (like SE) followed by spatial attention (like the attention gate), in one module.
> - An **edge-aware** skip: add `cv2.Canny`-style edge information, or a Sobel conv (Lesson 01 Part A!), to the level-1 skip.
> - A **multi-scale** decoder block: the `dilated` block in the decoder too (`block="dilated"` everywhere is expensive, so try it on the bottleneck plus the last decoder step).
> - **Deep supervision** as in UNet++ (Lesson 05).
>
> Then bring the results table and your plot, and we'll analyse them together.

---
## Cheat sheet: where to modify UNet

| I want to change... | setting | example |
|---|---|---|
| input/output size behaviour | `padding` | `"valid"` (paper) vs `"same"` |
| model size | `base_channels`, `channel_mult` | `32`, `1.5` |
| how deep it looks | `depth` | `4`, `6` |
| what each stage is | `block`, `bottleneck` | `"residual"`, `"dilated"` |
| training stability | `norm`, `act`, `dropout` | `"gn"`, `"gelu"`, `0.3` |
| downsampling | `down` | `"conv"` |
| upsampling | `up` | `"transpose"` |
| the skip connections | `merge`, `skip_gate` | `"add"`, `"attention"` |
| something new | your class in `BLOCKS` / `GATES` | Part 6 |